<h1>Chapter 5 - MCP</h1>
<i>Giving an Agent access to the Environment through Tool Usage</i>


<a href="..."><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="..."><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="..."><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](...)

---

This notebook is for Chapter 5 of the [An Illustrated Guide to AI Agents](...) book by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="...">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on Google Colab <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [ ]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

## 1 - Choosing Your LLM - `Gemma 3`

At the beginning of every chapter, we start by choosing the LLM that we want to use. In this notebook, we will explore how to enable tool calling for models that do not have this capability. Since this notebook continues from `chapter05.ipynb` we will be using `Gemma 3`.

In [1]:
import os
from illustrated_agents.chapters.ch2 import LLM

# Ollama through OpenAI API
llm = LLM(model="gemma3:12b", backend="openai", api_base="http://localhost:11434/v1/")

# Llama.cpp server
# llm = LLM(model="openai/gemma-3-12B-it-Q4_K_M", backend="litellm", api_base="http://localhost:8080")

# LM Studio
# llm = LLM(model="lm_studio/gemma-4-12B-it", backend="litellm", api_base="http://localhost:1234/v1")

# Google's Gemini / Gemma
# os.environ['GEMINI_API_KEY'] = "YOUR_API_KEY"
# llm = LLM(model="gemini/gemini-2.5-flash", backend="litellm", api_key=None)

## 2 - Adding Tools with **`MCP`**

As covered in the book, Model Context Protocol (MCP), is an incredible way of standardizing tool usage. In this bonus notebook, we will be building an MCP Server and Client so that we can easily run any tool for which an MCP Server exists. 

### 🔹 MCP **`Server`**

The MCP Server is a lightweight program that exposes APIs and tools via the MCP standard. These servers often connect to a specific data source or service. For instance, an MCP server might connect to all API endpoints of arXiv to search, load, and view academic papers.


![../images/ch5_mcp_server.png](../images/ch5_mcp_server.png)


To built the MCP Server, we only need to define our tools and add them to the MCP Server using the `mcp` package. The code for this server is stored in `src/illustrated_agents/chapters/ch5_mcp_server.py` and the MCP Client will run it whenever it feels it needs to access it. As such, there is no need to run it within the notebook but the code is fortunately rather straightforward. The following shows what is in `.../ch5_mcp_server.py`. Note that all we need to do to add a tool is to run `@mcp.tool()` to add a tool (a function) to the server:

```python
import requests
from mcp.server.fastmcp import FastMCP


# Initialize an MCP server
mcp = FastMCP("file_reader")


# Define and add our tools to the MCP server
@mcp.tool()
def read_markdown(path: str) -> str:
    """Read the content of a markdown file."""
    return requests.get(path).text


# Run the server
def main():
    mcp.run(transport="stdio")


if __name__ == "__main__":
    main()
```

### 🔹 MCP **`Client`**

Next up is the more complex component, the MCP **Client**. The **Client** is the bridge between the **LLM** (or host) and the **Server**. We can consider your **TinyAgent** to be the MCP **Host** in this example. 

![../images/ch5_mcp_client.png](../images/ch5_mcp_client.png)



Before we show you how to show the tools available, we first need to define the path to the server. As mentioned, that is in the source package (`illustrated-agents`) and can be accessed like so:

In [2]:
from pathlib import Path
import illustrated_agents

# Enable nest_asyncio for MCP compatibility in notebooks
import nest_asyncio
nest_asyncio.apply()

# This path is used to start the MCP server from the tools module
SERVER_PATH = str(Path(illustrated_agents.__file__).parent / "chapters" / "ch5_mcp_server.py")

To show which tools are available, we can create a small function called `list_tools` that will attempt to access the server:

In [3]:
import sys
from mcp.client.stdio import stdio_client, StdioServerParameters
from mcp import ClientSession


async def list_tools():
    # The parameters to start the MCP server
    server_params = StdioServerParameters(command=sys.executable, args=[SERVER_PATH])
    
    # Connect to the MCP server over stdio
    async with stdio_client(server_params, errlog=None) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            tools = await session.list_tools()
            for tool in tools.tools:
                print(f"Name: {tool.name}")
                print(f"Description: {tool.description}")
                print(f"Parameters: {tool.inputSchema}")
                print()

await list_tools()

Name: read_markdown
Description: Read the content of a markdown file.

    Args:
        path (str): The path to the markdown file.
    
Parameters: {'properties': {'path': {'title': 'Path', 'type': 'string'}}, 'required': ['path'], 'title': 'read_markdownArguments', 'type': 'object'}



Note how it loads the name of the function we created before, the docstring, and even the parameters of the function! As covered in the book, this is why it is so important to write proper docstrings as it will be the main thing that your LLM/Agent reads. 

Next up, let's see if we can use the tool and read the markdown of the BERTopic package (a topic modeling framework):

In [4]:
async def call_tool():
    # The parameters to start the MCP server
    server_params = StdioServerParameters(
        command=sys.executable,
        args=[SERVER_PATH],
    )
    
    # Connect to the MCP server over stdio
    async with stdio_client(server_params, errlog=None) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            result = await session.call_tool("read_markdown", {"path": "https://raw.githubusercontent.com/MaartenGr/BERTopic/refs/heads/master/README.md"})
            print(result.content[0].text)

await call_tool()

[![PyPI Downloads](https://static.pepy.tech/badge/bertopic)](https://pepy.tech/projects/bertopic)
[![PyPI - Python](https://img.shields.io/badge/python-v3.10+-blue.svg)](https://pypi.org/project/bertopic/)
[![Build](https://img.shields.io/github/actions/workflow/status/MaartenGr/BERTopic/testing.yml?branch=master)](https://github.com/MaartenGr/BERTopic/actions)
[![docs](https://img.shields.io/badge/docs-Passing-green.svg)](https://maartengr.github.io/BERTopic/)
[![PyPI - PyPi](https://img.shields.io/pypi/v/BERTopic)](https://pypi.org/project/bertopic/)
[![PyPI - License](https://img.shields.io/badge/license-MIT-green.svg)](https://github.com/MaartenGr/VLAC/blob/master/LICENSE)
[![arXiv](https://img.shields.io/badge/arXiv-2203.05794-<COLOR>.svg)](https://arxiv.org/abs/2203.05794)


# BERTopic

<img src="images/logo.png" width="35%" align="right" /> 

BERTopic is a topic modeling technique that leverages 🤗 transformers and c-TF-IDF to create dense clusters
allowing for easily interpretab

Yes, it does! The function works properly via MCP. These two functions are at the heart of our MCP **Client**, which can list the tools available and run them through the MCP **Server**.

Next, we will need to add these functions to the `Tools` class that you created in this chapter. To do so, we can inherit from that class and create a new one, namely `MCPTools(Tools)`:

In [5]:

import asyncio

from illustrated_agents.tools import Tools

class MCPTools(Tools):
    """MCP-based tool registry that extends Tools."""

    def __init__(self):
        super().__init__()
        self._load_mcp_tools()

    def _get_server_params(self):
        return StdioServerParameters(command=sys.executable, args=[SERVER_PATH])

    def _load_mcp_tools(self):
        """Fetch tools from MCP server and register them."""

        # Fetch the list of tools from the MCP server
        async def fetch():
            async with stdio_client(self._get_server_params(), errlog=None) as (read, write):
                async with ClientSession(read, write) as session:
                    await session.initialize()
                    tools = await session.list_tools()
                    return tools.tools

        mcp_tools = asyncio.run(fetch())

        # Register each MCP tool using parent's add_tool
        for tool in mcp_tools:
            self.add_tool(
                name=tool.name,
                func=self._make_tool_caller(tool.name),
                description=tool.description,
            )

    def _make_tool_caller(self, name: str):
        """Create a callable that invokes the MCP tool."""

        def caller(**kwargs):
            return asyncio.run(self._call_tool(name, kwargs))

        return caller

    async def _call_tool(self, name: str, args: dict) -> str:
        """Call a tool on the MCP server."""
        async with stdio_client(self._get_server_params(), errlog=None) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                result = await session.call_tool(name, args)
                return result.content[0].text

As always, let's go through some of these functions step-by-step:

In [6]:
from illustrated_agents.chapters.ch5 import run_mcp_tool_annotated
run_mcp_tool_annotated

Finally, we can use our newly created `MCPTools` in the `TinyAgent` of this chapter:

In [7]:
from illustrated_agents.memory import Memory
from illustrated_agents.chapters.ch5 import TinyAgent

# Create MCP-based tools (automatically loads tools from server)
tools = MCPTools()

# Create agent
memory = Memory()
agent = TinyAgent(llm=llm, tools=tools, memory=memory)

We can then test your MCP-enabled `TinyAgent` and see whether it can correctly read the markdown of our previous book [`Hands-On Large Language Models`](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models):

In [8]:
agent.memory.get_messages()

[{'role': 'system',
  'content': 'You are a helpful assistant.\n\n\n# Tools\n\nIf needed, you can only use the following tools to assist you in completing tasks:\n\n`read_markdown`: Read the content of a markdown file.\n\n    Args:\n        path (str): The path to the markdown file.\n    \n\nTo use a tool, respond with JSON: {"tool": "name", "kwargs": {"param": "value"}}\n'}]

In [9]:
query = "Read the file at 'https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/refs/heads/main/README.md'"
print(agent.run(query))

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\maart\AppData\Roaming\uv\python\cpython-3.12.9-windows-x86_64-none\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000001C18EC2A500> is already entered


OBSERVATION: ```json
{"tool": "read_markdown", "kwargs": {"path": "https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/refs/heads/main/README.md"}}
``` -> ﻿# Hands-On Large Language Models

<a href="https://www.linkedin.com/in/jalammar/"><img src="https://img.shields.io/badge/Follow%20Jay-blue.svg?logo=linkedin"></a>
<a href="https://www.linkedin.com/in/mgrootendorst/"><img src="https://img.shields.io/badge/Follow%20Maarten-blue.svg?logo=linkedin"></a>
<a href="https://www.deeplearning.ai/short-courses/how-transformer-llms-work/?utm_campaign=handsonllm-launch&utm_medium=partner"><img src="https://img.shields.io/badge/DeepLearning.AI%20Course-NEW!-&labelColor=black&color=red.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHZpZXdCb3g9IjAuMDAwMzY1MjgxIC0wLjAwMDE0MDE0MiAzMy4yOSAzMy4xNSI+Cgk8cGF0aCBkPSJNMTYuNjQzIDMzLjE0NWMtMy4yOTIgMC02LjUxLS45NzItOS4yNDYtMi43OTNhMTYuNTg4IDE2LjU4OCAwIDAxLTYuMTMtNy40MzhBMTYuNTA3IDE2LjUwNyAwIDAx

Great! Your one-step `TinyAgent` use the correct tool for the job. As we will cover in Chapter 6, we often would rather have an Agent that can do more than just run a tool once. In this particular example, it would be nice if your `TinyAgent` could follow it up with a summary of the README.md. We can do this by running by leveraging the [Reason and Act (ReAct)](https://openreview.net/pdf?id=WE_vluYUL-X) framework which autonomously performs as many sequential steps as necessary:

In [10]:
from illustrated_agents.planning import ReAct
from illustrated_agents.chapters.ch6 import TinyAgent

# Create MCP-based tools (automatically loads tools from server)
tools = MCPTools()

# Create agent
planner = ReAct()  # Will cover in chapter 6 and allows for multi-step reasoning
memory = Memory()
agent = TinyAgent(llm=llm, tools=tools, memory=memory, planner=planner)

We run the same query as before:

In [11]:
query = "Read the file at 'https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/refs/heads/main/README.md' and summarize its content in 3 sentences."
print(agent.run(query))

This repository contains code examples and explanations for the book 'Hands-On Large Language Models' by Jay Alammar and Maarten Grootendorst, which visually details the practical tools and concepts of LLMs. The book is available on multiple platforms and includes over 300 custom-made figures and covers topics from introductory concepts to advanced techniques like fine-tuning and multimodal models. It's praised by experts like Andrew Ng for its clarity and practical approach to understanding and using Large Language Models.


Your `TinyAgent` correctly read the file and then summarized it. Let's check the messages to get an understanding of how these MCP calls work:

In [12]:
agent.memory.get_messages()

[{'role': 'system',
  'content': 'You are a helpful AI agent.\n\n\n# ReAct (Reason and Act)\n\nYou are a ReAct agent that performs exactly ONE step per turn.\nMake sure to break down a given task into smaller steps and decide whether to use a tool or provide a final answer.\n\n## ReAct Format\n\nYou use the following format for each step:\n\nTHOUGHT: [Your reasoning about what to do next]\nACTION:\n{\n    "tool": "a_tool_name",\n    "kwargs": {"param": "value"},\n}\n\nAn observation will be provided after each action. You do not generate the observation yourself.\n\n## ReAct Completion\n\nTo provide the final answer to the task, use an action blob with "tool": "final_answer" tool. \nIt is the only way to complete the task, else you will be stuck on a loop. So your final output should look like this:\n\nACTION:\n{\n    "tool": "final_answer",\n    "kwargs": "insert your final answer here"\n}\n\nUse the `final_answer` tool when you are completely done with all subtasks and have the final

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

# What We Built

In this chapter, we covered how to add `MCP` to your `TinyAgent`. Doing so required creating `MCPTools` that inherited from `Tools`. As a result, this new class only needs to be added to `tools.py`: 

In [ ]:
from illustrated_agents.chapters.ch5 import what_we_built_mcp; what_we_built_mcp